# ChaosMesh Arena TRL Training Example (Levels 1-3)

This notebook demonstrates a curriculum-style loop for Levels 1-3.
It is intentionally simple and deterministic enough to use as a hackathon demo baseline.

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from statistics import mean
from typing import Any

from chaosmesh_arena.env import ChaosMeshArenaEnv
from chaosmesh_arena.models import ActionModel, ActionType, AgentRole, IncidentLevel

In [ ]:
def make_noop_action() -> ActionModel:
    # A low-risk baseline action so we can focus on curriculum mechanics.
    return ActionModel(
        agent=AgentRole.DIAGNOSTICS,
        action_type=ActionType.QUERY_METRICS,
        target='svc-api',
    )


def run_episode(level: IncidentLevel, max_steps: int = 20, demo_mode: bool = True) -> dict[str, Any]:
    env = ChaosMeshArenaEnv(level=level, demo_mode=demo_mode)
    obs, info = env.reset(options={'level': level.value})

    terminated = False
    truncated = False
    total_reward = 0.0

    for _ in range(max_steps):
        action = make_noop_action()
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += float(reward)
        if terminated or truncated:
            break

    final_state = env.state()
    unresolved = [i for i in final_state.active_incidents if i.status != 'resolved']

    return {
        'level': level.value,
        'episode_id': final_state.episode_id,
        'steps': final_state.step,
        'status': final_state.episode_status,
        'terminated': terminated,
        'truncated': truncated,
        'cumulative_reward': final_state.cumulative_reward,
        'unresolved_incident_count': len(unresolved),
        'sim_time_minutes': final_state.sim_time_minutes,
    }

In [ ]:
# Curriculum: run 2 episodes per level for Levels 1 -> 3.
curriculum = [IncidentLevel.LEVEL_1, IncidentLevel.LEVEL_2, IncidentLevel.LEVEL_3]
episodes_per_level = 2

results = []
for lvl in curriculum:
    for _ in range(episodes_per_level):
        results.append(run_episode(level=lvl, max_steps=25, demo_mode=True))

results

In [ ]:
# Lightweight aggregate view without requiring pandas.
by_level = {}
for row in results:
    lvl = row['level']
    by_level.setdefault(lvl, []).append(row)

summary = []
for lvl, rows in sorted(by_level.items()):
    summary.append({
        'level': lvl,
        'episodes': len(rows),
        'avg_steps': round(mean(r['steps'] for r in rows), 2),
        'avg_reward': round(mean(r['cumulative_reward'] for r in rows), 3),
        'avg_unresolved_incidents': round(mean(r['unresolved_incident_count'] for r in rows), 2),
        'termination_rate': round(sum(1 for r in rows if r['terminated']) / len(rows), 2),
    })

summary

## Next Steps

- Replace `make_noop_action()` with your policy agent output.
- Increase `episodes_per_level` for stable TRL metrics.
- Persist `results` to SQLite or experiment tracking for trend analysis.